In [85]:
import pandas as pd
from ydata_profiling import ProfileReport

In [86]:
# df = pd.read_csv(r"G:\Mi unidad\COMPARTIR_INFORMACION\AUTORIZACIONES\MUESTRA_ORACLE_2025\muestra_volantes_2025.txt",
#                  sep="|",
#                  encoding="latin1",
#                  dtype="str")
                 

In [87]:
# leer excel con multiples hojas
ruta_archivo = r"G:\Mi unidad\Requerimientos_Demanda\Requerimientos\2025-08-12\Malla Regulacion Consolidado.xlsx"

sheets = pd.ExcelFile(ruta_archivo).sheet_names
dfs_sheet = []
for sheet in sheets:
    df_sheet = pd.read_excel(ruta_archivo, sheet_name=sheet)
    df_sheet['sheet'] = sheet
    dfs_sheet.append(df_sheet)
df = pd.concat(dfs_sheet, ignore_index=True)
df['sheet'].value_counts()

sheet
C-13           36560
C-19           36560
Comparativo    32888
C-12           30257
C-10           29884
C-7            25308
Name: count, dtype: int64

In [88]:
columnas_texto = [
    "documento",
    "identificacion",
    "identificación",
    "codigo",
    "código",
    "cod",
    "cód",
    "dane",
    "ciudad",
    "nombre",
    "número de autorización",
    "año",
    "mes",
    "anio",
    "aut",
    "cum"
]

columnas_numericas_flotantes = [
    "costo",
    "total",
    "valor",
    "vr",
    "tope"
]

In [89]:
columnas_nulas = []

for columna in df.columns:
    if df[columna].isnull().all():
        columnas_nulas.append(columna)
        print(f"Columna {columna} está completamente vacía")

In [90]:
def _infer_decimal_sep(s: pd.Series) -> str:
    
    sample = s.dropna().astype(str).str.strip().head(1000)
    comma = sample.str.contains(",", regex=False).any()
    dot = sample.str.contains(".", regex=False).any()
    return "," if comma and not dot else "."

In [91]:
# revisar si alguna palabra de la lista columnas_numericas_flotantes está en el nombre de la columna
columnas_posibles_flotantes = []
for col in df.select_dtypes('object').columns:
    contiene_num = any(palabra in col.lower() for palabra in columnas_numericas_flotantes)
    no_contiene_texto = all(palabra_texto not in col.lower() for palabra_texto in columnas_texto)
    if contiene_num and no_contiene_texto:
        columnas_posibles_flotantes.append(col)
columnas_posibles_flotantes

[]

In [92]:
for columna in df.select_dtypes('object').columns:
    if columna in columnas_nulas or columnas_texto:
        continue

    if _infer_decimal_sep(df[columna]) == ",":
        try:
            df[columna] = df[columna].str.replace(',', '.', regex=False)
            df[columna] = pd.to_numeric(df[columna], errors='raise')
            df[columna] = df[columna].round(2)
            print(f"Columna {columna} convertida a float.")
        except Exception as e:
            print(f"Error al convertir la columna {columna}: {e}")
            continue
    else:
        try:
            df[columna] = pd.to_numeric(df[columna], errors='raise')
            df[columna] = df[columna].round(2)
            print(f"Columna {columna} convertida a float.")
        except Exception as e:
            print(f"Error al convertir la columna {columna}: {e}")

In [93]:
columnas_no_codigo = []

for columna in df.select_dtypes('object').columns:
    if not any(palabra in columna.lower() for palabra in columnas_texto):
        columnas_no_codigo.append(columna)
        
columnas_no_codigo

['Expediente',
 'ACTO ADMINISTRATIVO',
 'ID MR',
 'Producto_Farmacia',
 'Principio_Activo',
 'sheet']

In [94]:
# Detectar columnas numericas int
for columna in df.select_dtypes('object').columns:
    if df[columna].str.isnumeric().all() and columna not in columnas_nulas and columna in columnas_no_codigo:
        try:
            df[columna].fillna(0, inplace=True)  # Rellenar NaN con 0
            df[columna] = pd.to_numeric(df[columna], downcast='integer', errors='raise')
            print(f"Columna {columna} convertida a int.")
        except:
            print(f"Error al convertir la columna {columna} a int.")
            continue

In [95]:
formatos_python_fechas = [
    "%Y-%m-%d",
    "%d/%m/%Y",
    "%d-%m-%Y",
    "%Y/%m/%d"
]

In [96]:
for columna in df.select_dtypes('object').columns:
    if df[columna].str.contains(r"[/-]", na=False).any() or 'fecha' in columna.lower() or 'date' in columna.lower():
        for formato in formatos_python_fechas:
            try:
                    df[columna] = df[columna].str.slice(0, 10)
                    df[columna] = df[columna].loc[df[columna].str.contains("0001", na=False) == False]
                    df[columna] = pd.to_datetime(df[columna], format=formato, errors='raise')
                    print(f"Columna {columna} convertida a fecha con formato {formato}.")
                    break  # Si se convierte correctamente, salimos del bucle
            except Exception as e:
                if 'fecha' in columna.lower():
                    print(f"Columna {columna} no pudo ser convertida a fecha con formato {formato}.")
                continue

In [97]:
df.dtypes

CUM                                         object
Precio Unidad Minima de Dispensacion       float64
Expediente                                  object
ACTO ADMINISTRATIVO                         object
Valor CUM + Intermediacion                 float64
Valor CUM                                  float64
% Intermediacion                           float64
Precio por Unidad  de Regulación C-7       float64
ID MR                                       object
FC                                         float64
Producto_Farmacia                           object
Principio_Activo                            object
Código                                      object
CANTIDAD                                     int64
Valor UMD sin intermediacion               float64
Precio por Unidad  de Regulación C-7.1     float64
sheet                                       object
Precio por Unidad  de Regulación C-19      float64
Precio por Unidad  de Regulación C-12      float64
Precio por Unidad  de Regulació

In [98]:
for columna in df.select_dtypes('object').columns:
    cantidad_unicos = df[columna].nunique(dropna=False)
    total_registros = df[columna].count()
    cardinalidad = cantidad_unicos / total_registros if total_registros > 0 else 0
    if cardinalidad < 0.01 and cardinalidad > 0 and cantidad_unicos < 300:
        print(f"{cardinalidad:.2f} - Columna: {columna} - Cantidad únicos: {cantidad_unicos} - Total registros: {total_registros}")
        try:
            df[columna] = df[columna].astype('category')
            print(f"Columna {columna} convertida a categoría.")
        except:
            continue    

0.00 - Columna: ACTO ADMINISTRATIVO - Cantidad únicos: 8 - Total registros: 191457
Columna ACTO ADMINISTRATIVO convertida a categoría.
0.00 - Columna: ID MR - Cantidad únicos: 94 - Total registros: 20820
Columna ID MR convertida a categoría.
0.00 - Columna: sheet - Cantidad únicos: 6 - Total registros: 191457
Columna sheet convertida a categoría.


In [99]:
def contar_precision_numerica(serie: pd.Series):
    # transformar a entero
    serie_entera = serie.dropna().astype(int)
    max_entero = serie_entera.astype(str).str.len().max()
    return max_entero + 2

In [100]:
def detectar_tipo_columna_oracle(columna):
    if df[columna].dtype == 'int32' or df[columna].dtype == 'int64' or df[columna].dtype == 'int16' or df[columna].dtype == 'int8':
        return f"NUMBER({df[columna].astype(str).str.len().max()})"
    elif df[columna].dtype == 'float32' or df[columna].dtype == 'float64':
        return f"NUMBER({contar_precision_numerica(df[columna])}, 2)"
    elif df[columna].dtype == 'datetime64[ns]':
        return "DATE"
    elif df[columna].dtype == 'category' or df[columna].dtype == 'object':
        if len(df[columna].str.len().value_counts()) == 1:
            return f"CHAR({df[columna].astype(str).str.len().max()})"
        return f"VARCHAR2({df[columna].astype(str).str.len().max()})"
    else:
        print(f"Tipo de columna {columna} no reconocido: {df[columna].dtype}")
        return "VARCHAR2(255)"  # Valor por defecto para tipos no reconocidos
    
        

In [101]:
def analizar_patrones(df, columna):
    if df[columna].dtype == 'category':
        return df[columna].value_counts().head(20).index.tolist()
    elif 'int' in str(df[columna].dtype) or 'float' in str(df[columna].dtype):
        return f"min: {df[columna].describe()['min'].round(1)} - max: {df[columna].describe()['max'].round(1)} - mean: {df[columna].describe()['mean'].round(1)} - std: {df[columna].describe()['std'].round(1)}"
    elif df[columna].dtype == 'datetime64[ns]' and df[columna].notnull().any():
        return f"min: {df[columna].min().strftime('%d/%m/%Y')} - max: {df[columna].max().strftime('%d/%m/%Y')}"
    else:
        return ''

In [102]:
dict_columnas_oracle_esquema = {}

for columna in df.columns:
    # Obtenemos el tipo de dato para Oracle
    tipo_dato = detectar_tipo_columna_oracle(columna)
    
    # Verificamos si la columna tiene valores nulos (True o False)
    tiene_nulos = df[columna].isnull().any()
    
    
    if tiene_nulos:
        obligatoria = 'No Obligatorio'
    else:
        obligatoria = 'Obligatorio'
    
    # Guardamos ambas informaciones en un diccionario anidado
    dict_columnas_oracle_esquema[columna] = {
        'Tipo': tipo_dato,
        'longitud': df[columna].astype(str).str.len().max(),
        'valores_permitidos': analizar_patrones(df, columna),
        'obligatoria': obligatoria,
    }


In [103]:
dict_columnas_oracle_esquema


{'CUM': {'Tipo': 'VARCHAR2(10)',
  'longitud': 10,
  'valores_permitidos': '',
  'obligatoria': 'No Obligatorio'},
 'Precio Unidad Minima de Dispensacion': {'Tipo': 'NUMBER(11, 2)',
  'longitud': 12,
  'valores_permitidos': 'min: 2.0 - max: 790370445.5 - mean: 317047.6 - std: 7474578.7',
  'obligatoria': 'Obligatorio'},
 'Expediente': {'Tipo': 'VARCHAR2(15)',
  'longitud': 15,
  'valores_permitidos': '',
  'obligatoria': 'Obligatorio'},
 'ACTO ADMINISTRATIVO': {'Tipo': 'VARCHAR2(33)',
  'longitud': 33,
  'valores_permitidos': ['CIRCULAR 19 DE 2024 (Articulo 1)',
   'Circular 10 de 2020, Artículo 4',
   'Circular 7 de 2018, Artículo 6',
   'CIRCULAR 04 DE 2016 (ARTICULO 2)',
   'Circular 7 de 2018, Artículo 5',
   'CIRCULAR 06 DEL 2013 (ARTICULO 1)',
   'CIRCULAR 3 DEL 2017 (ARTICULO 7)',
   'Circular 13 de 2022, Artículo 3'],
  'obligatoria': 'Obligatorio'},
 'Valor CUM + Intermediacion': {'Tipo': 'NUMBER(12, 2)',
  'longitud': 18,
  'valores_permitidos': 'min: 10.5 - max: 1896889069.2

In [104]:
# Generar diccionario en formato excel
df_esquema = pd.DataFrame.from_dict(dict_columnas_oracle_esquema, orient='index')
df_esquema.index.name = 'Columna'
df_esquema

,Tipo,longitud,valores_permitidos,obligatoria
Columna,,,,
CUM,VARCHAR2(10),10,,No Obligatorio
Precio Unidad Minima de Dispensacion,"NUMBER(11, 2)",12,min: 2.0 - max: 790370445.5 - mean: 317047.6 -...,Obligatorio
Expediente,VARCHAR2(15),15,,Obligatorio
ACTO ADMINISTRATIVO,VARCHAR2(33),33,"[CIRCULAR 19 DE 2024 (Articulo 1), Circular 10...",Obligatorio
Valor CUM + Intermediacion,"NUMBER(12, 2)",18,min: 10.5 - max: 1896889069.2 - mean: 1587053....,Obligatorio
Valor CUM,"NUMBER(12, 2)",18,min: 9.9 - max: 1832743062.0 - mean: 1529765.4...,Obligatorio
% Intermediacion,"NUMBER(3, 2)",5,min: 0.0 - max: 0.1 - mean: 0.1 - std: 0.0,Obligatorio
Precio por Unidad de Regulación C-7,"NUMBER(9, 2)",10,min: 0.6 - max: 7573700.2 - mean: 19841.7 - st...,No Obligatorio
ID MR,VARCHAR2(7),7,"[458b, 959a, 989a, 543a, 938a, 771b, 75b, 75a,...",No Obligatorio


In [105]:
df_esquema.to_excel(r"G:\Mi unidad\COMPARTIR_INFORMACION\AUTORIZACIONES\MUESTRA_ORACLE_2025\esquema_muestra_oracle.xlsx", index=True)

In [106]:
dict_columnas_oracle_esquema

{'CUM': {'Tipo': 'VARCHAR2(10)',
  'longitud': 10,
  'valores_permitidos': '',
  'obligatoria': 'No Obligatorio'},
 'Precio Unidad Minima de Dispensacion': {'Tipo': 'NUMBER(11, 2)',
  'longitud': 12,
  'valores_permitidos': 'min: 2.0 - max: 790370445.5 - mean: 317047.6 - std: 7474578.7',
  'obligatoria': 'Obligatorio'},
 'Expediente': {'Tipo': 'VARCHAR2(15)',
  'longitud': 15,
  'valores_permitidos': '',
  'obligatoria': 'Obligatorio'},
 'ACTO ADMINISTRATIVO': {'Tipo': 'VARCHAR2(33)',
  'longitud': 33,
  'valores_permitidos': ['CIRCULAR 19 DE 2024 (Articulo 1)',
   'Circular 10 de 2020, Artículo 4',
   'Circular 7 de 2018, Artículo 6',
   'CIRCULAR 04 DE 2016 (ARTICULO 2)',
   'Circular 7 de 2018, Artículo 5',
   'CIRCULAR 06 DEL 2013 (ARTICULO 1)',
   'CIRCULAR 3 DEL 2017 (ARTICULO 7)',
   'Circular 13 de 2022, Artículo 3'],
  'obligatoria': 'Obligatorio'},
 'Valor CUM + Intermediacion': {'Tipo': 'NUMBER(12, 2)',
  'longitud': 18,
  'valores_permitidos': 'min: 10.5 - max: 1896889069.2

In [107]:
esquema_oracle = "CREATE TABLE nombre_tabla ("

for columna, dict in dict_columnas_oracle_esquema.items():
    esquema_oracle += f'    "{columna}" {dict["Tipo"]},\n'
esquema_oracle = esquema_oracle.rstrip(', ') + ');'
print(esquema_oracle)

CREATE TABLE nombre_tabla (    "CUM" VARCHAR2(10),
    "Precio Unidad Minima de Dispensacion" NUMBER(11, 2),
    "Expediente" VARCHAR2(15),
    "ACTO ADMINISTRATIVO" VARCHAR2(33),
    "Valor CUM + Intermediacion" NUMBER(12, 2),
    "Valor CUM" NUMBER(12, 2),
    "% Intermediacion" NUMBER(3, 2),
    "Precio por Unidad  de Regulación C-7" NUMBER(9, 2),
    "ID MR" VARCHAR2(7),
    "FC" NUMBER(8, 2),
    "Producto_Farmacia" VARCHAR2(10),
    "Principio_Activo" VARCHAR2(10),
    "Código" VARCHAR2(9),
    "CANTIDAD" NUMBER(5),
    "Valor UMD sin intermediacion" NUMBER(11, 2),
    "Precio por Unidad  de Regulación C-7.1" NUMBER(9, 2),
    "sheet" VARCHAR2(10),
    "Precio por Unidad  de Regulación C-19" NUMBER(11, 2),
    "Precio por Unidad  de Regulación C-12" NUMBER(11, 2),
    "Precio por Unidad  de Regulación C-13" NUMBER(11, 2),
    "Variacion" NUMBER(5, 2),
    "Precio por Unidad  de Regulación C-13.1" NUMBER(11, 2),
    "Unnamed: 18" NUMBER(4, 2),
);


In [108]:
# create dict to a csv
dict_columnas_oracle_esquema_df = pd.DataFrame.from_dict(dict_columnas_oracle_esquema, orient='index', columns=['Tipo'])
dict_columnas_oracle_esquema_df.index.name = 'Columna'
dict_columnas_oracle_esquema_df.to_csv("esquema_oracle.csv", encoding='utf-8-sig')

In [109]:
# profile = ProfileReport(
#     df,
#     title="Perfilamiento Volantes 2025",
#     minimal=True
# )

In [110]:
# export profile to json file
# profile.to_file("perfilamiento_volantes_2025.json")